In [ ]:
# === STANDART COLAB KURULUM HÜCRESİ ===
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
REPO_URL = "https://github.com/Oldzhay34/receipts-and-interactions.git"
BRANCH   = "b/1.8-scaling"
try:
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
url = REPO_URL.replace("https://", f"https://{TOKEN}@") if TOKEN else REPO_URL

if not os.path.exists("/content/repo"):
    !git clone -q -b {BRANCH} {url} /content/repo
else:
    !cd /content/repo && git fetch -q && git checkout -q {BRANCH} && git pull -q

%cd /content/repo
!pip install -q -r requirements.txt && pip install -q -e .

if "/content/repo/src" not in sys.path:
    sys.path.insert(0, "/content/repo/src")

os.environ["DATA_ROOT"] = "/content/drive/MyDrive/CME4403/data"
from recsys.config import load_config
CFG = load_config()
print("commit:", os.popen("git rev-parse --short HEAD").read().strip())

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from recsys.io.artifact_store import ArtifactStore
from recsys.features.scaling import (
    WeightedCombinedDistance,
    build_numeric_scaler,
    compare_scalers,
)
from recsys.features.distance import JaccardDistance
from recsys.viz.plots import hist_box

outputs_root = getattr(CFG, "outputs_dir", Path(os.environ["DATA_ROOT"]) / "outputs")
store = ArtifactStore(outputs_root)
loaded = store.load("loaded")
recipes = loaded["recipes"]

numeric_cols = [
    "minutes",
    "n_steps",
    "n_ingredients",
    "calories",
    "total_fat_pdv",
    "sugar_pdv",
    "sodium_pdv",
    "protein_pdv",
    "saturated_fat_pdv",
    "carbohydrates_pdv",
]
numeric_cols = [c for c in numeric_cols if c in recipes.columns]
print(f"Recipes: {recipes.shape} | Numeric cols: {len(numeric_cols)}")

In [ ]:
scaler_summary = compare_scalers(recipes, numeric_cols)
display(scaler_summary)

In [ ]:
mm_scaler = build_numeric_scaler({"method": "minmax", "use_log1p": True})
zs_scaler = build_numeric_scaler({"method": "zscore", "use_log1p": True})

raw_num = recipes[numeric_cols].apply(pd.to_numeric, errors="coerce").dropna()
mm_df = pd.DataFrame(mm_scaler.fit_transform(raw_num), columns=numeric_cols, index=raw_num.index)
zs_df = pd.DataFrame(zs_scaler.fit_transform(raw_num), columns=numeric_cols, index=raw_num.index)

figures = {
    "fig_08_01_calories_raw": hist_box(raw_num["calories"], title="Calories (Raw)"),
    "fig_08_02_calories_log1p_minmax": hist_box(mm_df["calories"], title="Calories (Log1p + MinMax)"),
    "fig_08_03_calories_log1p_zscore": hist_box(zs_df["calories"], title="Calories (Log1p + Z-Score)"),
    "fig_08_04_minutes_raw": hist_box(raw_num["minutes"], title="Minutes (Raw)"),
    "fig_08_05_minutes_log1p_minmax": hist_box(mm_df["minutes"], title="Minutes (Log1p + MinMax)"),
}
print(f"Generated {len(figures)} scaling figures.")

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

sample_recipes = recipes.loc[mm_df.index[:5]]
mlb = MultiLabelBinarizer(sparse_output=True)
X_ing_sample = mlb.fit_transform(sample_recipes["ingredients"])
X_num_sample = mm_df.loc[sample_recipes.index, ["minutes", "n_ingredients", "calories"]].to_numpy()

combined = WeightedCombinedDistance(ingredient_metric=JaccardDistance(), numeric_weight=0.2)
D_sample = combined.pairwise((X_ing_sample, X_num_sample), (X_ing_sample, X_num_sample))
dist_df = pd.DataFrame(
    np.round(D_sample, 4),
    index=[f"recipe_{i}" for i in range(5)],
    columns=[f"recipe_{i}" for i in range(5)],
)
display(dist_df)

In [ ]:
saved = store.save(
    stage="scaling",
    objects={
        "scaler_summary": scaler_summary,
        "combined_distance_sample": dist_df,
        **figures,
    },
    inputs=[loaded],
    params={
        "default_method": "minmax",
        "use_log1p": True,
        "numeric_weight": 0.2,
        "n_numeric_cols": len(numeric_cols),
    },
)

for fig in figures.values():
    plt.close(fig)

print("Saved scaling artifact:", saved.version)
print("\n=== SCALER SUMMARY (minutes & calories) ===")
print(
    scaler_summary[scaler_summary["feature"].isin(["minutes", "calories", "n_ingredients"])]
    .to_string(index=False)
)